# 06 — Modelado completo y búsqueda de hiperparámetros

Interfaz interactiva para el pipeline orquestado en `src/search.py`. Equivalente a `scripts/03_run_hyperparameter_search.py`, pero con celdas separadas para inspeccionar paso a paso.

**Reglas obligatorias (codificadas en el módulo)**

- Split 80/20 estricto por `case_id` con `make_train_test_group_split_with_coverage`.
- `beat_type` está prohibido como predictor (bloqueado por `assert_no_forbidden_features`).
- CV interna por grupo (`StratifiedGroupKFold` cuando es viable; `GroupKFold` como fallback).
- El test se evalúa **una sola vez** al final, después de fijar hiperparámetros con CV en train.
- Métrica primaria: `f1_macro`. Complementarias: `precision_macro`, `recall_macro`, `accuracy`, `balanced_accuracy`, `f1_weighted`.

**Pre-requisitos**

1. `scripts/01_download_all_available_ecg.py` ha descargado los ECG deseados a `data/raw/vitaldb_waveforms/`.
2. `scripts/02_build_features_all_windows.py` ha generado los parquets de features para 1.2 s, 2.0 s y 5.0 s en `data/processed/`.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config
from src.evaluation import confusion_matrix_with_totals, per_class_report
from src.search import (
    MODEL_REGISTRY,
    PRIMARY_SCORING,
    WindowRunConfig,
    assemble_best_hyperparameters_table,
    assemble_class_support_table,
    assemble_comparison_table,
    assemble_missing_classes_table,
    pick_best_overall,
    run_one_window,
)
from src.utils import get_logger, set_seed

set_seed(config.RANDOM_SEED)
logger = get_logger("nb06")
sns.set_theme(context="notebook", style="whitegrid")

## 2. Configuración del run

Ajusta `MODELS_TO_RUN`, `WINDOWS_TO_RUN`, `N_ITER`, `N_SPLITS` según el alcance que necesites. Para una corrida rápida, usa `N_ITER=3` y `N_SPLITS=2`.

In [ ]:
WINDOW_FILENAMES = {
    1.2: "features_w1p2s.parquet",
    2.0: "features_w2p0s.parquet",
    5.0: "features_w5p0s.parquet",
}

MODELS_TO_RUN = list(MODEL_REGISTRY.keys())
WINDOWS_TO_RUN = [1.2, 2.0, 5.0]
N_ITER = 30
N_SPLITS = 5
TEST_SIZE = 0.2
N_JOBS = -1

print("Modelos:", MODELS_TO_RUN)
print("Ventanas:", WINDOWS_TO_RUN)
print("n_iter:", N_ITER, "| n_splits:", N_SPLITS, "| test_size:", TEST_SIZE)

## 3. Ejecución del pipeline por ventana

In [ ]:
window_results = []
for w in WINDOWS_TO_RUN:
    parquet_path = config.PROCESSED_DIR / WINDOW_FILENAMES[w]
    if not parquet_path.exists():
        logger.warning("Falta %s. Salto la ventana %.1fs.", parquet_path, w)
        continue
    cfg = WindowRunConfig(
        window_seconds=w,
        parquet_path=parquet_path,
        models_to_run=MODELS_TO_RUN,
        n_iter=N_ITER,
        n_splits=N_SPLITS,
        test_size=TEST_SIZE,
        random_state=config.RANDOM_SEED,
        n_jobs=N_JOBS,
    )
    logger.info("=== Ventana %.1fs ===", w)
    wr = run_one_window(cfg)
    window_results.append(wr)

print(f"Ventanas procesadas: {len(window_results)}")

## 4. Diagnóstico del split por ventana

In [ ]:
split_rows = []
for wr in window_results:
    info = wr["split_info"]
    split_rows.append({
        "window_seconds": wr["window_seconds"],
        "chosen_seed": info["chosen_seed"],
        "n_classes_covered": info["n_classes_covered"],
        "n_total_classes": info["n_total_classes"],
        "actual_test_fraction": round(info["actual_test_fraction"], 3),
        "classes_only_in_train": info["classes_only_in_train"],
        "classes_only_in_test": info["classes_only_in_test"],
        "cv_splitter": wr["cv_info"]["splitter"],
        "cv_n_splits": wr["cv_info"]["n_splits_effective"],
    })
pd.DataFrame(split_rows)

## 5. Comparación de modelos

In [ ]:
comparison_df = assemble_comparison_table(window_results)
comparison_df.round(3)

In [ ]:
ok = comparison_df.loc[comparison_df["status"] == "ok"].copy()
pivot = ok.pivot_table(index="model", columns="window_seconds", values=f"test_{PRIMARY_SCORING}", aggfunc="first").sort_index()
pivot.round(3)

## 6. Mejores hiperparámetros por (ventana, modelo)

In [ ]:
bhp = assemble_best_hyperparameters_table(window_results)
bhp

## 7. Mejor modelo global y reporte por clase

In [ ]:
winner = pick_best_overall(comparison_df, primary=f"test_{PRIMARY_SCORING}")
if winner is None:
    print("Sin ganador válido.")
else:
    w_win = float(winner["window_seconds"])
    m_win = str(winner["model"])
    print(f"Mejor modelo: {m_win} @ ventana={w_win:.1f}s — test_{PRIMARY_SCORING}={winner[f'test_{PRIMARY_SCORING}']:.3f}")
    winning_wr = next(wr for wr in window_results if abs(wr["window_seconds"] - w_win) < 1e-9)
    winning_mr = next(mr for mr in winning_wr["models"] if mr["model"] == m_win and mr.get("status") == "ok")
    y_test = winning_wr["y_test"]
    y_pred = winning_mr["y_pred_test"]
    print("\nReporte por clase en test:")
    print(per_class_report(y_test, y_pred).round(3).to_string())
    print("\nMatriz de confusión absoluta + totales:")
    print(confusion_matrix_with_totals(y_test, y_pred).to_string())

## 8. Matriz de confusión (visual)

In [ ]:
from sklearn.metrics import confusion_matrix

if winner is None:
    print("Sin ganador válido; nada que graficar.")
else:
    labels = sorted(set(pd.Series(y_test).unique()) | set(pd.Series(y_pred).unique()), key=str)
    cm = confusion_matrix(y_test, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(1.0 + len(labels), 0.8 + 0.8 * len(labels)))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels, cbar=False, ax=ax)
    ax.set_title(f"Matriz de confusión absoluta — {m_win} @ {w_win:.1f}s")
    ax.set_xlabel("Predicho")
    ax.set_ylabel("Real")
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

## 9. Notas

- Para correr esto desde la línea de comandos (idéntica lógica), usar `python scripts/03_run_hyperparameter_search.py`. Acepta `--debug`, `--models`, `--windows`, `--n-iter`, `--n-splits`.
- El script persiste automáticamente todos los CSVs y figuras requeridos en `reports/`.
- Antes de interpretar las métricas, revisar la sección 4 de este notebook: si una clase está ausente de train o de test, la métrica macro arrastra un cero por construcción.